# Phase 2 — Condition 2: MONAI Pretrained (Frozen) Encoder

**Goal:** Test whether replacing the from-scratch (collapsed) Swin encoder with a *pretrained* 3D SwinUNETR encoder breaks the representational collapse and lets the fusion gate actually use the MRI modality.

**Baseline to beat (Condition 1, as-submitted):** gate weight on MRI = 0.0000; MRI-zeroed accuracy = full accuracy; image-feature similarity mean=1.0, std=0.0 (total collapse).

**Success criterion:** (a) image-feature similarity std clearly > 0; (b) gate assigns non-trivial weight to MRI; (c) zeroing MRI input measurably drops test metrics.

**This notebook is self-contained and logged.** It (1) verifies data before any model runs, (2) loads pretrained weights *loudly* (no silent random fallback), (3) runs an image-only encoder sanity probe, (4) trains the full tri-modal model with the pretrained frozen encoder, (5) runs the identical diagnostic as Condition 1, (6) appends results to the Phase 2 lab log.

> Run cells top to bottom. STOP if any **ASSERT** cell fails — it means a precondition is wrong and continuing would waste a run.

## 0. Environment setup

In [ ]:
# Mount Drive and install pinned dependencies
from google.colab import drive
drive.mount('<DRIVE_ROOT>', force_remount=True)

# Pin MONAI to a known version to reduce state-dict key drift when loading pretrained weights.
!pip install -q 'monai==1.3.0' nibabel torchio 'antspyx==0.4.2' imbalanced-learn
print('\n✅ Setup cell done. Check above for any pip errors.')

## 1. Configuration & imports

All paths and the condition name live here. `CONDITION_NAME` tags every output file so nothing mixes between conditions.

In [ ]:
import os, json, datetime, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchio as tio
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from monai.networks.nets import SwinUNETR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'No GPU detected. Set Runtime > Change runtime type > GPU before running.'

# ---- Condition identity ----
CONDITION_NAME = 'condition2_monai_pretrained_frozen'

# ---- Paths (match the canonical Tri_Model_Inititative1_3 notebook) ----
DRIVE_PROJECT_PATH = '<DRIVE_ROOT>/ADNI_NewDS/'
RESULTS_DIRECTORY  = os.path.join(DRIVE_PROJECT_PATH, 'results')
PROCESSED_MRI_DIRECTORY = os.path.join(RESULTS_DIRECTORY, 'processed_mri_scans_swin')
SPLIT_IDS_PATH     = os.path.join(RESULTS_DIRECTORY, 'patient_id_splits.npz')
CLEANED_DATA_PATH  = os.path.join(RESULTS_DIRECTORY, 'project_data_cleaned.csv')
BIOMARKER_SEQUENCES_PATH = os.path.join(RESULTS_DIRECTORY, 'preprocessed_biomarker_sequences.npy')

# ---- Phase 2 output area (kept separate from Phase 1) ----
PHASE2_DIR  = os.path.join(DRIVE_PROJECT_PATH, 'Phase2_Collapse_Study')
COND_DIR    = os.path.join(PHASE2_DIR, 'results', CONDITION_NAME)
LAB_LOG_PATH= os.path.join(PHASE2_DIR, 'phase2_lab_log.jsonl')
os.makedirs(COND_DIR, exist_ok=True)

# ---- Pretrained weights ----
PRETRAINED_URL = 'https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/model_swinvit.pt'
PRETRAINED_LOCAL = os.path.join(PHASE2_DIR, 'model_swinvit.pt')

IMG_SIZE = (96, 96, 96)
FEATURE_SIZE = 48              # vit_feature_size = 48*16 = 768 (matches baseline)
NUM_CLASSES = 3
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print('✅ Config set. Condition:', CONDITION_NAME)

## 2. DATA VERIFICATION (run before anything else)

This is the cell that prevents wasted runs. It confirms the splits, the MRI directory, the clinical CSV, and the biomarker file all exist, load, and have the expected shapes. **If any ASSERT fails, stop and fix the path — do not continue.**

In [ ]:
# --- 2.1 Splits ---
assert os.path.exists(SPLIT_IDS_PATH), f'Missing splits: {SPLIT_IDS_PATH}'
sp = np.load(SPLIT_IDS_PATH, allow_pickle=True)
pids_train, pids_val, pids_test = sp['pids_train'], sp['pids_val'], sp['pids_test']
labels_train, labels_val, labels_test = sp['labels_train'], sp['labels_val'], sp['labels_test']
print(f'Splits: train={len(pids_train)} val={len(pids_val)} test={len(pids_test)}')
print('Test class distribution (CN,MCI,Dem):', np.bincount(labels_test, minlength=3))
assert len(pids_train)==130 and len(pids_test)==29, 'Split sizes differ from the audited baseline (130/28/29).'

# --- 2.2 MRI directory + one sample volume ---
assert os.path.isdir(PROCESSED_MRI_DIRECTORY), f'Missing MRI dir: {PROCESSED_MRI_DIRECTORY}'
sample_path = os.path.join(PROCESSED_MRI_DIRECTORY, f'{pids_train[0]}_processed.npy')
assert os.path.exists(sample_path), f'Missing sample volume: {sample_path}'
vol = np.load(sample_path)
print(f'Sample MRI {pids_train[0]}: shape={vol.shape} dtype={vol.dtype} '
      f'min={vol.min():.3f} max={vol.max():.3f} mean={vol.mean():.3f}')
assert vol.ndim == 3, f'Expected 3D volume, got shape {vol.shape}'

# --- 2.3 Confirm ALL referenced volumes exist (catch missing files now, not mid-train) ---
all_pids = list(pids_train)+list(pids_val)+list(pids_test)
missing = [p for p in all_pids if not os.path.exists(os.path.join(PROCESSED_MRI_DIRECTORY, f'{p}_processed.npy'))]
assert not missing, f'{len(missing)} MRI files missing, e.g. {missing[:5]}'
print(f'✅ All {len(all_pids)} MRI volumes present.')

# --- 2.4 Clinical CSV ---
import pandas as pd
assert os.path.exists(CLEANED_DATA_PATH), f'Missing clinical CSV: {CLEANED_DATA_PATH}'
clin_df = pd.read_csv(CLEANED_DATA_PATH)
FEATURE_COLUMNS = ['AGE','PTGENDER','PTEDUCAT','APOE4','MMSE','ADAS13',
                   'RAVLT_immediate','RAVLT_learning','RAVLT_forgetting','FAQ']
missing_cols = [c for c in FEATURE_COLUMNS if c not in clin_df.columns]
assert not missing_cols, f'CSV missing columns: {missing_cols}'
print(f'Clinical CSV: {clin_df.shape[0]} rows, {clin_df["PTID"].nunique()} patients.')

# --- 2.5 Biomarker sequences ---
assert os.path.exists(BIOMARKER_SEQUENCES_PATH), f'Missing biomarker file: {BIOMARKER_SEQUENCES_PATH}'
bio_raw = np.load(BIOMARKER_SEQUENCES_PATH, allow_pickle=True)
print(f'Biomarker array: type={type(bio_raw)} shape/{getattr(bio_raw,"shape","?")}')
print('✅ DATA VERIFICATION PASSED. Safe to proceed.')

## 3. Build clinical & biomarker sequence tensors

Replicates the baseline's per-patient sequence construction so the only thing changing in this condition is the **image encoder**.

In [ ]:
# Clinical sequences: one variable-length sequence per patient (grouped by PTID)
patient_clinical = {pid: torch.tensor(g[FEATURE_COLUMNS].values, dtype=torch.float32)
                    for pid, g in clin_df.groupby('PTID')}

# Biomarker sequences: stored as a dict-like or array aligned to PTID.
# We coerce into {pid: tensor}. Inspect then adapt if structure differs.
bio_obj = bio_raw.item() if (hasattr(bio_raw,'dtype') and bio_raw.dtype==object and bio_raw.ndim==0) else bio_raw
if isinstance(bio_obj, dict):
    patient_biomarker = {k: torch.tensor(np.asarray(v), dtype=torch.float32) for k,v in bio_obj.items()}
    print('Biomarker stored as dict keyed by PTID. n=', len(patient_biomarker))
else:
    raise RuntimeError('Biomarker structure is not a dict; print bio_obj and adapt this cell before continuing.')

# Verify every split PID has both sequences
for p in all_pids:
    assert p in patient_clinical,  f'No clinical sequence for {p}'
    assert p in patient_biomarker, f'No biomarker sequence for {p}'
BIO_DIM = next(iter(patient_biomarker.values())).shape[-1]
CLIN_DIM = len(FEATURE_COLUMNS)
print(f'✅ Sequences built. clinical feat dim={CLIN_DIM}, biomarker feat dim={BIO_DIM}')

## 4. Dataset & dataloaders

In [ ]:
from torch.nn.utils.rnn import pad_sequence

train_tf = tio.Compose([tio.RandomFlip(axes='LR'), tio.Resize(IMG_SIZE),
                        tio.ZNormalization(masking_method=tio.ZNormalization.mean)])
eval_tf  = tio.Compose([tio.Resize(IMG_SIZE),
                        tio.ZNormalization(masking_method=tio.ZNormalization.mean)])

class TriModalDataset(Dataset):
    def __init__(self, pids, labels, transform):
        self.pids=list(pids); self.labels=torch.tensor(labels, dtype=torch.long); self.tf=transform
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        pid=self.pids[i]
        vol=np.load(os.path.join(PROCESSED_MRI_DIRECTORY, f'{pid}_processed.npy'))
        subj=tio.Subject(mri=tio.ScalarImage(tensor=torch.tensor(vol,dtype=torch.float32).unsqueeze(0)))
        mri=self.tf(subj).mri.tensor.squeeze(0)
        return mri, patient_clinical[pid], patient_biomarker[pid], self.labels[i]

def collate(batch):
    mris, clins, bios, ys = zip(*batch)
    return (torch.stack(mris),
            pad_sequence(clins, batch_first=True),
            pad_sequence(bios,  batch_first=True),
            torch.stack(ys))

BATCH = 4  # raised from baseline's 2-3 (audit H4); frozen encoder keeps memory low
train_loader = DataLoader(TriModalDataset(pids_train,labels_train,train_tf), batch_size=BATCH, shuffle=True,  collate_fn=collate, num_workers=2)
val_loader   = DataLoader(TriModalDataset(pids_val,  labels_val,  eval_tf),  batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=2)
test_loader  = DataLoader(TriModalDataset(pids_test, labels_test, eval_tf),  batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=2)
print('✅ Loaders ready. Batches: train',len(train_loader),'val',len(val_loader),'test',len(test_loader))

## 5. Download & load pretrained encoder (LOUD — no silent fallback)

This directly fixes audit bug **C4**. The weights are CT-pretrained (domain gap to MRI), and MONAI versions are known to mismatch some state-dict keys, so we load with `strict=False` **and assert that a meaningful fraction of encoder keys actually loaded.** If too few match, the cell raises — it will not quietly proceed on random weights.

In [ ]:
from monai.networks.utils import copy_model_state

# 5.1 Download once to Drive
if not os.path.exists(PRETRAINED_LOCAL):
    print('Downloading pretrained SwinViT weights...')
    import urllib.request; urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_LOCAL)
print('Pretrained file:', PRETRAINED_LOCAL, f'({os.path.getsize(PRETRAINED_LOCAL)/1e6:.1f} MB)')

# 5.2 Build a fresh SwinUNETR and pull its swinViT encoder
_swin = SwinUNETR(img_size=IMG_SIZE, in_channels=1, out_channels=1, feature_size=FEATURE_SIZE, use_checkpoint=True)
encoder = _swin.swinViT

# 5.3 Load weights, counting matches
ckpt = torch.load(PRETRAINED_LOCAL, map_location='cpu')
state = ckpt.get('state_dict', ckpt.get('model', ckpt))
# strip common prefixes
clean = { k.replace('module.','').replace('swinViT.','').replace('swin_vit.',''): v for k,v in state.items() }
enc_sd = encoder.state_dict()
matched = {k:v for k,v in clean.items() if k in enc_sd and v.shape==enc_sd[k].shape}
frac = len(matched)/max(1,len(enc_sd))
print(f'Encoder keys: {len(enc_sd)} | matched from checkpoint: {len(matched)} ({frac:.1%})')

# 5.4 LOUD assertion (the anti-C4 guard)
assert frac >= 0.5, (f'Only {frac:.1%} of encoder keys loaded from pretrained file. '
                     'Refusing to continue on a near-random encoder. Inspect key names above.')
encoder.load_state_dict(matched, strict=False)
print(f'✅ Pretrained encoder loaded ({frac:.1%} of keys). NOT random init.')

## 6. Tri-modal model (pretrained **frozen** encoder + dual LSTM + gated fusion)

Architecture matches the audited baseline (`AdvancedMultiModalModel`): image projection → 256→128, two LSTM branches, a softmax fusion gate over the three 128-d modality features, then a classifier. The **only** change vs Condition 1 is that the encoder weights are pretrained, and we expose `unfreeze_encoder` so a later condition can fine-tune.

In [ ]:
class LSTMBranch(nn.Module):
    def __init__(self, in_dim, hidden=128, out=128):
        super().__init__()
        self.lstm=nn.LSTM(in_dim, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.fc=nn.Linear(hidden, out); self.relu=nn.ReLU()
    def forward(self,x):
        _,(h,_)=self.lstm(x); return self.relu(self.fc(h[-1]))

class TriModalPretrained(nn.Module):
    def __init__(self, encoder, clin_dim, bio_dim, num_classes=3, feature_size=48, unfreeze_encoder=False):
        super().__init__()
        vit_dim = feature_size*16  # 768
        self.encoder = encoder
        self.unfreeze_encoder = unfreeze_encoder
        for p in self.encoder.parameters(): p.requires_grad = unfreeze_encoder
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.img_proj = nn.Sequential(nn.Linear(vit_dim,256), nn.ReLU(), nn.Linear(256,128))
        self.clin_branch = LSTMBranch(clin_dim, out=128)
        self.bio_branch  = LSTMBranch(bio_dim,  out=128)
        self.fusion_gate = nn.Sequential(nn.Linear(128*3,128), nn.ReLU(), nn.Linear(128,3), nn.Softmax(dim=1))
        self.classifier  = nn.Sequential(nn.LayerNorm(128), nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64,num_classes))
    def _img_feat(self, mri):
        x = mri.unsqueeze(1)
        ctx = torch.enable_grad() if self.unfreeze_encoder else torch.no_grad()
        with ctx:
            feat = self.encoder(x)[-1]
        feat = self.pool(feat).flatten(1)
        return self.img_proj(feat)
    def forward(self, mri, clin, bio, return_gate=False):
        fi=self._img_feat(mri); fc=self.clin_branch(clin); fb=self.bio_branch(bio)
        stack=torch.stack([fi,fc,fb], dim=1)              # [B,3,128]
        w=self.fusion_gate(torch.cat([fi,fc,fb], dim=1))  # [B,3]
        fused=(stack*w.unsqueeze(-1)).sum(1)              # [B,128]
        logits=self.classifier(fused)
        return (logits, w) if return_gate else logits

model = TriModalPretrained(encoder, CLIN_DIM, BIO_DIM, NUM_CLASSES, FEATURE_SIZE, unfreeze_encoder=False).to(device)
n_train=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model built. Trainable params: {n_train:,} (encoder frozen)')

## 7. Image-only encoder probe (diagnostic isolation)

Before training the full model, check whether the **pretrained encoder alone** produces class-separable features. We freeze the encoder, extract pooled features for train/test, fit a quick logistic regression, and read test accuracy.

- If **> ~0.45** (well above the 0.52 majority-class / 0.33 random baselines being beaten meaningfully): the encoder yields usable signal → any later gate-zeroing is a *fusion* issue.
- If **≈ chance**: the encoder itself is still the bottleneck for this data, informing whether Condition 4 (custom SSL) is needed.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

@torch.no_grad()
def extract_img_feats(loader):
    model.eval(); X=[]; Y=[]
    for mri,_,_,y in tqdm(loader, desc='extract'):
        f=model._img_feat(mri.to(device)).cpu().numpy()
        X.append(f); Y.append(y.numpy())
    return np.concatenate(X), np.concatenate(Y)

Xtr,ytr=extract_img_feats(train_loader); Xte,yte=extract_img_feats(test_loader)
sc=StandardScaler().fit(Xtr)
clf=LogisticRegression(max_iter=2000, class_weight='balanced').fit(sc.transform(Xtr), ytr)
probe_acc=accuracy_score(yte, clf.predict(sc.transform(Xte)))
# collapse check on the raw image features
Fn=Xte/ (np.linalg.norm(Xte,axis=1,keepdims=True)+1e-8)
sim=Fn@Fn.T; off=sim[~np.eye(len(sim),dtype=bool)]
print(f'IMAGE-ONLY probe test accuracy: {probe_acc:.4f}')
print(f'Image-feature similarity: mean={off.mean():.4f} std={off.std():.4f}  (Condition1 was mean=1.0 std=0.0)')
probe_results={'probe_img_only_acc':float(probe_acc),'img_sim_mean':float(off.mean()),'img_sim_std':float(off.std())}

## 8. Train the full tri-modal model

Frozen encoder, so only the projection, LSTMs, gate, and classifier train. Early-stopping on validation MCC (more reliable than accuracy on this imbalanced set).

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
EPOCHS=50; PATIENCE=12
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-5)
sched=CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-7)
cw=compute_class_weight('balanced', classes=np.unique(labels_train), y=labels_train)
crit=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float).to(device))

best_mcc=-1; best_state=None; wait=0; hist=[]
BEST_PATH=os.path.join(COND_DIR,'best_model.pth')
for ep in range(EPOCHS):
    model.train(); tot=0
    for mri,clin,bio,y in tqdm(train_loader, desc=f'Epoch {ep+1}/{EPOCHS}', leave=False):
        mri,clin,bio,y=mri.to(device),clin.to(device),bio.to(device),y.to(device)
        opt.zero_grad(); loss=crit(model(mri,clin,bio), y); loss.backward(); opt.step(); tot+=loss.item()
    # val
    model.eval(); P=[];T=[]
    with torch.no_grad():
        for mri,clin,bio,y in val_loader:
            out=model(mri.to(device),clin.to(device),bio.to(device))
            P+=out.argmax(1).cpu().tolist(); T+=y.tolist()
    vmcc=matthews_corrcoef(T,P); vacc=accuracy_score(T,P); hist.append({'epoch':ep+1,'val_mcc':vmcc,'val_acc':vacc})
    print(f'Epoch {ep+1}: val_acc={vacc:.4f} val_mcc={vmcc:.4f}')
    if vmcc>best_mcc: best_mcc=vmcc; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; torch.save(best_state,BEST_PATH); wait=0; print('  ⭐ new best (val_mcc)')
    else:
        wait+=1
        if wait>=PATIENCE: print('  🛑 early stop'); break
    sched.step()
if best_state: model.load_state_dict(best_state)
print(f'✅ Training done. Best val_mcc={best_mcc:.4f}')

## 9. Diagnostic (identical instrument to Condition 1)

The decisive comparison. Same three measurements as the baseline: mean gate weights, leave-one-modality-out ablation, image-feature similarity. Anything that differs from Condition 1 here is the *effect of the pretrained encoder*.

In [ ]:
def gmean(cm):
    gs=[]
    for i in range(cm.shape[0]):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-(tp+fn+fp)
        se=tp/(tp+fn) if (tp+fn)>0 else 0; sp=tn/(tn+fp) if (tn+fp)>0 else 0; gs.append((se*sp)**0.5)
    return float(np.mean(gs))

model.eval(); gates=[]; imgfeats=[]
with torch.no_grad():
    for mri,clin,bio,y in test_loader:
        mri,clin,bio=mri.to(device),clin.to(device),bio.to(device)
        _,w=model(mri,clin,bio,return_gate=True); gates.append(w.cpu().numpy())
        imgfeats.append(model._img_feat(mri).cpu().numpy())
gates=np.concatenate(gates); mean_gates=gates.mean(0)
print('='*56); print('(A) MEAN GATE WEIGHTS'); print(f'  MRI={mean_gates[0]:.4f}  Clinical={mean_gates[1]:.4f}  Bio={mean_gates[2]:.4f}')

def ablate(zero=None):
    P=[];T=[]
    with torch.no_grad():
        for mri,clin,bio,y in test_loader:
            mri,clin,bio=mri.to(device),clin.to(device),bio.to(device)
            if zero=='mri': mri=torch.zeros_like(mri)
            if zero=='clin': clin=torch.zeros_like(clin)
            if zero=='bio': bio=torch.zeros_like(bio)
            out=model(mri,clin,bio); P+=out.argmax(1).cpu().tolist(); T+=y.tolist()
    cm=confusion_matrix(T,P,labels=[0,1,2])
    return accuracy_score(T,P), matthews_corrcoef(T,P), gmean(cm)
print('='*56); print('(B) LEAVE-ONE-MODALITY-OUT')
abl={}
for tag,z in [('full',None),('mri',  'mri'),('clin','clin'),('bio','bio')]:
    a,m,g=ablate(z); abl[tag]={'acc':a,'mcc':m,'gmean':g}
    print(f'  {tag:5s}  acc={a:.4f}  mcc={m:.4f}  gmean={g:.4f}')

F=np.concatenate(imgfeats); Fn=F/(np.linalg.norm(F,axis=1,keepdims=True)+1e-8); sim=Fn@Fn.T; off=sim[~np.eye(len(sim),dtype=bool)]
print('='*56); print('(C) IMAGE-FEATURE SIMILARITY'); print(f'  mean={off.mean():.4f}  std={off.std():.4f}')
diag={'gate':{'mri':float(mean_gates[0]),'clin':float(mean_gates[1]),'bio':float(mean_gates[2])},
      'ablation':abl,'img_sim_mean':float(off.mean()),'img_sim_std':float(off.std())}

## 10. Verdict & append to Phase 2 lab log

Applies the success criterion automatically and appends one JSON line to the shared lab log so every condition is comparable.

In [ ]:
full=diag['ablation']['full']; mri0=diag['ablation']['mri']
recovered = (diag['img_sim_std']>0.01) and (diag['gate']['mri']>0.10) and ((full['acc']-mri0['acc'])>0.01)
verdict = 'RECOVERED imaging branch' if recovered else 'imaging branch still inert'
print('VERDICT:', verdict)
print(f"  similarity std {diag['img_sim_std']:.4f} (>0.01?)  gate_mri {diag['gate']['mri']:.4f} (>0.10?)  "
      f"acc drop when MRI zeroed {full['acc']-mri0['acc']:+.4f} (>0.01?)")

entry={'timestamp':datetime.datetime.utcnow().isoformat(),'condition':CONDITION_NAME,
       'change':'MONAI CT-pretrained SwinUNETR encoder, frozen; batch=4; dual-LSTM+gated fusion unchanged',
       'best_val_mcc':float(best_mcc),'probe':probe_results,'diagnostic':diag,
       'verdict':verdict,'recovered':bool(recovered),
       'baseline_ref':{'gate_mri':0.0,'img_sim_std':0.0,'full_acc':0.8966}}
with open(LAB_LOG_PATH,'a') as f: f.write(json.dumps(entry)+'\n')
json.dump(entry, open(os.path.join(COND_DIR,'result.json'),'w'), indent=2)
print('✅ Logged to', LAB_LOG_PATH)
print('\nPaste this whole cell output back to Claude for interpretation.')